# U-Net segmentation with uncertainty

This notebook reproduces the four-panel display from the referenced notebook: **input**, **ground truth**, **inference**, and an **uncertainty mask**. It assumes a `segmentation_models_pytorch` U-Net checkpoint at `models/unet_best_model.pth`.

The uncertainty panel is the pixelwise standard deviation across Monte Carlo dropout predictions. A meaningful MC-dropout map requires dropout layers to have been present while the checkpoint was trained. If the loaded model has no dropout layers, the notebook raises a clear error instead of silently returning a zero mask.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# Defaults mirror Train2d-backbone.py; set CME_DATA_ROOT to override on another server.
PROJECT_ROOT = Path.cwd()
CHECKPOINT_PATH = PROJECT_ROOT / 'models' / 'unet_best_model.pth'
DATA_ROOT = Path(os.environ.get('CME_DATA_ROOT', '/data/CME_reconstruction/cme_silhouette/processed_dataset_new'))
CSV_PATH = DATA_ROOT / 'file_list.csv'
TARGET_SIZE = (832, 832)  # Must match training.
THRESHOLD = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {DEVICE}; checkpoint: {CHECKPOINT_PATH}')

In [ ]:
def pad_to_target(array, target_size=TARGET_SIZE):
    """Center-pad a (H, W) array; fail rather than crop a mismatched dataset."""
    array = np.nan_to_num(np.asarray(array), nan=0.0).astype(np.float32)
    if array.ndim != 2:
        raise ValueError(f'Expected a 2-D array, got {array.shape}.')
    height, width = array.shape
    target_h, target_w = target_size
    if height > target_h or width > target_w:
        raise ValueError(f'Input {array.shape} is larger than TARGET_SIZE={target_size}.')
    pad_h, pad_w = target_h - height, target_w - width
    return np.pad(array, ((pad_h // 2, pad_h - pad_h // 2),
                          (pad_w // 2, pad_w - pad_w // 2)), mode='constant')

def load_sample(idx):
    """Load this repository's sample format: {'input': image, 'mask': mask}."""
    files = pd.read_csv(CSV_PATH)
    if not 0 <= idx < len(files):
        raise IndexError(f'idx must be in [0, {len(files) - 1}].')
    sample_path = DATA_ROOT / files.iloc[idx]['file_path']
    sample = np.load(sample_path, allow_pickle=True).item()
    image = pad_to_target(sample['input'])
    mask = np.asarray(sample['mask']).squeeze()
    mask = pad_to_target((mask > 0).astype(np.float32))
    return image, mask, sample_path

assert CSV_PATH.exists(), f'Dataset index not found: {CSV_PATH}'

In [ ]:
# This architecture matches the U-Net setup in Train2d-backbone.py.
# If the remote checkpoint used another encoder/architecture, change these values
# to its training configuration before loading it.
model = smp.Unet(
    encoder_name='resnet18', encoder_weights=None,
    in_channels=1, classes=1, activation=None,
).to(DEVICE)

assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
state_dict = {key.removeprefix('module.'): value for key, value in state_dict.items()}
model.load_state_dict(state_dict, strict=True)
model.eval()
print('Checkpoint loaded successfully.')

In [ ]:
def enable_mc_dropout(module):
    """Enable dropout while preserving eval-mode BatchNorm statistics."""
    module.eval()
    dropout_layers = [child for child in module.modules()
                      if isinstance(child, (nn.Dropout, nn.Dropout2d, nn.Dropout3d, nn.AlphaDropout))]
    if not dropout_layers:
        raise RuntimeError(
            'No dropout layers were found. MC dropout requires a U-Net trained with dropout. '
            'Use the matching dropout-enabled architecture/checkpoint; do not add dropout only at inference.'
        )
    for layer in dropout_layers:
        layer.train()
    return len(dropout_layers)

@torch.inference_mode()
def mc_predict(input_tensor, n_samples=100):
    if n_samples < 2:
        raise ValueError('n_samples must be at least 2 to calculate a standard deviation.')
    enable_mc_dropout(model)
    probabilities = torch.stack([torch.sigmoid(model(input_tensor)) for _ in range(n_samples)])
    model.eval()
    return probabilities.mean(dim=0), probabilities.std(dim=0, unbiased=False)

In [ ]:
def displayTestcaseWithUncertainty(idx, n_samples=100, string=''):
    """Show input, ground truth, mean inference, and MC-dropout uncertainty."""
    image, ground_truth, sample_path = load_sample(idx)
    input_tensor = torch.from_numpy(image)[None, None].to(DEVICE)

    model.eval()
    with torch.inference_mode():
        deterministic_probability = torch.sigmoid(model(input_tensor))[0, 0].cpu().numpy()
    mean_probability, uncertainty = mc_predict(input_tensor, n_samples=n_samples)
    uncertainty = uncertainty[0, 0].cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize=(18, 5), constrained_layout=True)
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('input')
    axes[1].imshow(ground_truth, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('ground truth')
    axes[2].imshow(deterministic_probability, cmap='gray', vmin=0, vmax=1)
    axes[2].contour(deterministic_probability >= THRESHOLD, levels=[0.5], colors='lime', linewidths=0.7)
    axes[2].set_title('inference (probability)')
    uncertainty_plot = axes[3].imshow(uncertainty, cmap='plasma', vmin=0)
    axes[3].set_title(f'uncertainty (MC std, n={n_samples})')
    fig.colorbar(uncertainty_plot, ax=axes[3], fraction=0.046, pad=0.04, label='probability std. dev.')
    for axis in axes:
        axis.axis('off')
    fig.suptitle(string or f'idx={idx} — {sample_path.name}')
    plt.show()
    return {'input': image, 'ground_truth': ground_truth,
            'inference': deterministic_probability, 'mean_probability': mean_probability[0, 0].cpu().numpy(),
            'uncertainty_mask': uncertainty}


In [ ]:
# Pick any valid index from file_list.csv. The returned dictionary includes the raw uncertainty mask.
result = displayTestcaseWithUncertainty(idx=0, n_samples=100, string='CME U-Net uncertainty')
uncertainty_mask = result['uncertainty_mask']
# np.save('uncertainty_mask_idx_0.npy', uncertainty_mask)  # optional export
